# Data Setup

This notebook requires access to the shared dataset folder for phase recognition.

Before running the pipeline, please complete the following steps:

1. Open the shared Google Drive folder link provided for this task.
2. Add the shared folder to the root of your **MyDrive** as a shortcut.
3. Keep the folder name unchanged as **`phase_recognition`**.

After mounting Google Drive in Colab, the dataset is expected to be available at:

`/content/drive/MyDrive/phase_recognition`

**Shared folder link:**  
`https://drive.google.com/drive/folders/1zlW2PCKx1OrfQBOxMUCzln36k_G79Thc?usp=share_link`

If the folder is not placed at this location, the subsequent pipeline will not be able to load the data correctly.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip -q install -U transformers accelerate datasets peft bitsandbytes qwen-vl-utils
!pip uninstall -y torchao

In [9]:
import os
import json
import torch
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, Qwen2_5_VLModel
from qwen_vl_utils import process_vision_info
import cv2
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
import time
import random
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import copy
from matplotlib.patches import Patch

# build ground truth jsonl

In [4]:
def get_target_frame_indices(num_frames_in_clip, fps, stride_sec=10):
    # Sample one target frame every `stride_sec` seconds.
    frame_indices = []
    sec = 0
    while True:
        fid = int(round(sec * stride_sec * fps))
        if fid >= num_frames_in_clip:
            break
        frame_indices.append(fid)
        sec += 1
    return frame_indices


def build_one_video_phase_row(mp4_path, json_path, sample, stride_sec=10):
    # Build one JSONL row containing sampled frame indices and phase labels for a single video.
    annotated_frame_ids = sorted(sample["annotated_frame_ids_local"])
    annotations_in_clip = sample["annotations_in_clip"]
    fps = float(sample["fps"])
    num_frames_in_clip = int(sample["num_frames_in_clip"])

    if len(annotated_frame_ids) == 0:
        return None

    target_frame_indices = get_target_frame_indices(
        num_frames_in_clip=num_frames_in_clip,
        fps=fps,
        stride_sec=stride_sec
    )

    phase_labels = []

    for target_fid in target_frame_indices:
        nearest_fid = min(annotated_frame_ids, key=lambda x: abs(x - target_fid))
        ann_list = annotations_in_clip[str(nearest_fid)]

        phase = ann_list[0]["phase"] if len(ann_list) > 0 else -1
        phase_labels.append(int(phase))

    row = {
        "video": mp4_path,
        "clip_json_path": json_path,
        "fps": fps,
        "num_frames_in_clip": num_frames_in_clip,
        "sample_stride_sec": int(stride_sec),
        "frame_indices": [int(x) for x in target_frame_indices],
        "phase_labels": phase_labels
    }
    return row


def process_split(data_dir, gt_name="gt_10s.jsonl", sample_stride_sec=10):
    out_path = os.path.join(data_dir, gt_name)

    if not os.path.exists(data_dir):
        print(f"[WARNING] split dir not found: {data_dir}")
        return

    files = sorted(os.listdir(data_dir))
    mp4_files = [f for f in files if f.endswith(".mp4") and "_clip_" in f]

    rows = []

    for mp4_name in mp4_files:
        base = os.path.splitext(mp4_name)[0]
        json_name = base + ".json"

        mp4_path = os.path.join(data_dir, mp4_name)
        json_path = os.path.join(data_dir, json_name)

        if not os.path.exists(json_path):
            print(f"[WARNING] missing json for: {mp4_path}")
            continue

        with open(json_path, "r", encoding="utf-8") as f:
            sample = json.load(f)

        if sample["num_annotated_frames"] == 0:
            print(f"[SKIP] no annotations: {json_path}")
            continue

        row = build_one_video_phase_row(
            mp4_path=mp4_path,
            json_path=json_path,
            sample=sample,
            stride_sec=sample_stride_sec
        )

        if row is not None:
            rows.append(row)

    with open(out_path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"saved: {out_path}")
    print(f"num videos in {data_dir}: {len(rows)}")


data_path = "/content/drive/MyDrive/phase_recognition"
training_dir = os.path.join(data_path, "training")
testing_dir = os.path.join(data_path, "testing")
process_split(training_dir)
process_split(testing_dir)

saved: /content/drive/MyDrive/phase_recognition/training/gt_10s.jsonl
num videos in /content/drive/MyDrive/phase_recognition/training: 17
saved: /content/drive/MyDrive/phase_recognition/testing/gt_10s.jsonl
num videos in /content/drive/MyDrive/phase_recognition/testing: 3


# zeroshot

In [14]:
# Utilities for zero-shot phase prediction post-processing:
# 1) safely parse raw model output as JSON,
# 2) validate whether the parsed JSON follows the expected phase-segment schema,
# 3) convert predicted phase segments into sampled frame-wise phase labels
#    so they can be aligned with ground truth for evaluation.

def safe_parse_json(x):
    if isinstance(x, dict):
        return x
    if not isinstance(x, str):
        return None
    x = x.strip()
    try:
        return json.loads(x)
    except Exception:
        return None


def schema_valid_phase_segments(obj):
    if not isinstance(obj, dict):
        return False
    segs = obj.get("phase_segments")
    if not isinstance(segs, list):
        return False
    for seg in segs:
        if not isinstance(seg, dict):
            return False
        required = {"start_frame", "end_frame", "phase"}
        if not required.issubset(seg.keys()):
            return False
    return True


def segments_to_sampled_phase_labels(segments, frame_indices):
    pred_labels = []
    for fid in frame_indices:
        assigned = -1
        for seg in segments:
            s = int(seg["start_frame"])
            e = int(seg["end_frame"])
            p = int(seg["phase"])
            if s <= fid <= e:
                assigned = p
                break
        pred_labels.append(assigned)
    return pred_labels

In [15]:
test_jsonl = "/content/drive/MyDrive/phase_recognition/testing/gt_10s.jsonl"
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
pred_out = "/content/drive/MyDrive/phase_recognition/outputs/zeroshot_predictions.jsonl"

PRIOR_RULE = (
    "Use the following prior information and label mappings. "
    "These mappings are provided only to explain the meaning of each integer ID. "
    "In the final JSON output, you must use integer IDs only. "
    "Do not output category names, text labels, or descriptions in the final JSON. "
)

PHASE_PRIOR = (
    "Phase ID mapping: "
    "0=Preparation; "
    "1=Calot triangle dissection; "
    "2=Clipping and cutting; "
    "3=Gallbladder dissection; "
    "4=Gallbladder retraction; "
    "5=Cleaning and coagulation; "
    "6=Gallbladder packaging. "
)

JSON_ONLY_RULE = (
    "Do not output markdown. "
    "Do not output code fences. "
    "Do not output explanations. "
    "Output JSON only. "
)

dataset = load_dataset("json", data_files=test_jsonl)["train"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

def build_phase_prompt(num_frames_in_clip: int) -> str:
    return (
        "<video>\n"
        f"{PRIOR_RULE}"
        f"{PHASE_PRIOR}"
        f"The video has frames indexed from 0 to {num_frames_in_clip - 1}. "
        "Return only a valid JSON object with exactly one key: 'phase_segments'. "
        "The value of 'phase_segments' must be a list of segments that cover the whole video "
        "from frame 0 to the last frame. "
        "Each segment must contain exactly these keys: "
        "'start_frame', 'end_frame', 'phase'. "
        "The segments must be continuous, non-overlapping, and sorted by time. "
        "The 'phase' value must be an integer ID only, using the phase ID mapping above. "
        "Do not use phase names or words in the final JSON. "
        "Do not return a list directly. "
        f"{JSON_ONLY_RULE}"
        "Return a JSON object of the form: "
        "{\"phase_segments\": [{\"start_frame\": 0, \"end_frame\": 100, \"phase\": 1}]}"
    )



def predict_zeroshot(sample, model, processor):
    video_path = sample["video"]
    num_frames_in_clip = int(sample["num_frames_in_clip"])
    prompt = build_phase_prompt(num_frames_in_clip).replace("<video>\n", "").replace("<video>", "").strip()

    if not video_path.startswith("file://"):
        video_path_for_msg = "file://" + video_path
    else:
        video_path_for_msg = video_path

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": video_path_for_msg,
                    "fps": 0.25,
                    "min_pixels": 256 * 256,
                    "max_pixels": 512 * 512,
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        return_video_kwargs=True
    )

    fixed_video_kwargs = {}
    for k, v in video_kwargs.items():
        if isinstance(v, list) and len(v) == 1:
            fixed_video_kwargs[k] = v[0]
        else:
            fixed_video_kwargs[k] = v

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        return_tensors="pt",
        **fixed_video_kwargs
    )

    for k, v in inputs.items():
        if isinstance(v, torch.Tensor):
            inputs[k] = v.to(model.device)

    model.eval()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
        )

    input_token_len = inputs["input_ids"].shape[1]
    new_tokens = generated_ids[:, input_token_len:]
    raw_pred = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    parsed_pred = safe_parse_json(raw_pred)

    return raw_pred, parsed_pred, prompt


rows = []

for i in range(len(dataset)):
    print(f"running zero-shot phase task {i+1}/{len(dataset)}")
    sample = dataset[i]
    raw_pred, parsed_pred, prompt = predict_zeroshot(sample,model,processor)

    gt_frame_indices = sample["frame_indices"]
    gt_phase_labels = sample["phase_labels"]

    pred_structured = None
    if schema_valid_phase_segments(parsed_pred):
        pred_phase_labels = segments_to_sampled_phase_labels(
            parsed_pred["phase_segments"],
            gt_frame_indices
        )
        pred_structured = {
            "frame_indices": gt_frame_indices,
            "phase_labels": pred_phase_labels
        }

    out_row = {
        "video": sample["video"],
        "clip_json_path": sample.get("clip_json_path", None),
        "prompt": prompt,
        "ground_truth": {
            "frame_indices": gt_frame_indices,
            "phase_labels": gt_phase_labels
        },
        "prediction": pred_structured,
        "raw_prediction": raw_pred
    }

    rows.append(out_row)

with open(pred_out, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("saved:", pred_out)

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

running zero-shot phase task 1/3
running zero-shot phase task 2/3
running zero-shot phase task 3/3
saved: /content/drive/MyDrive/phase_recognition/outputs/zeroshot_predictions.jsonl


# finetune

In [12]:
# Zero-shot phase recognition setup:
# load the test JSONL, define the phase prompt rules,
# and initialize the quantized Qwen2.5-VL model and processor.

train_jsonl = "/content/drive/MyDrive/phase_recognition/training/gt_10s.jsonl"
test_jsonl = "/content/drive/MyDrive/phase_recognition/testing/gt_10s.jsonl"
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
save_dir = "/content/drive/MyDrive/phase_recognition/outputs"

num_epochs = 8
lr = 1e-4
num_phase_classes = 7
batch_size = 1
max_frames_per_video = 90
device = "cuda" if torch.cuda.is_available() else "cpu"

grad_clip_norm = 1.0
adam_eps = 1e-6

# Data utilities for phase recognition:
# - read JSONL metadata,
# - check video accessibility,
# - load sampled RGB frames from videos,
# - sample frame indices at a fixed temporal stride (optionally with random offset),
# - assign phase labels using the nearest annotated frame,
# - wrap everything into a Dataset for training and inference.

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def can_open_video(video_path):
    if not os.path.exists(video_path):
        return False
    cap = cv2.VideoCapture(video_path)
    ok = cap.isOpened()
    cap.release()
    return ok


def read_frames_rgb(video_path, frame_indices, max_retries=3, sleep_sec=1.0):
    last_error = None

    for attempt in range(max_retries):
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            last_error = RuntimeError(f"Failed to open video: {video_path} | attempt={attempt+1}")
            cap.release()
            time.sleep(sleep_sec)
            continue

        frames = []
        ok_all = True

        for frame_index in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            ok, frame = cap.read()
            if not ok or frame is None:
                ok_all = False
                last_error = RuntimeError(
                    f"Failed to read frame {frame_index} from {video_path} | attempt={attempt+1}"
                )
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()

        if ok_all:
            return frames

        time.sleep(sleep_sec)

    raise last_error


def get_target_frame_indices_with_offset(num_frames_in_clip, fps, stride_sec=10, offset_sec=0.0):
    frame_indices = []
    t = float(offset_sec)
    while True:
        fid = int(round(t * fps))
        if fid >= num_frames_in_clip:
            break
        frame_indices.append(fid)
        t += stride_sec
    return frame_indices


def build_phase_labels_from_sample(sample, target_frame_indices):
    annotated_frame_ids = sorted(sample["annotated_frame_ids_local"])
    annotations_in_clip = sample["annotations_in_clip"]

    phase_labels = []
    for target_fid in target_frame_indices:
        nearest_fid = min(annotated_frame_ids, key=lambda x: abs(x - target_fid))
        ann_list = annotations_in_clip[str(nearest_fid)]
        phase = ann_list[0]["phase"] if len(ann_list) > 0 else -1
        phase_labels.append(int(phase))
    return phase_labels


class TaskAPhaseDataset(Dataset):
    def __init__(
        self,
        jsonl_path,
        max_frames_per_video=None,
        num_phase_classes=7,
        validate=False,
        sample_stride_sec=10,
        random_offset_sec=True,
    ):
        raw_rows = load_jsonl(jsonl_path)
        self.rows = []
        self.max_frames_per_video = max_frames_per_video
        self.num_phase_classes = num_phase_classes
        self.validate = validate
        self.sample_stride_sec = sample_stride_sec
        self.random_offset_sec = random_offset_sec

        for row in raw_rows:
            video_path = row["video"]
            clip_json_path = row.get("clip_json_path", None)

            if validate:
                if not can_open_video(video_path):
                    print("[SKIP] bad video:", video_path)
                    continue

                if clip_json_path is None or not os.path.exists(clip_json_path):
                    print("[SKIP] missing clip_json_path:", video_path)
                    continue

                with open(clip_json_path, "r", encoding="utf-8") as f:
                    sample = json.load(f)

                annotated_frame_ids = sorted(sample["annotated_frame_ids_local"])
                if len(annotated_frame_ids) == 0:
                    print("[SKIP] no annotations:", video_path)
                    continue

            self.rows.append(row)

        print("num usable rows:", len(self.rows))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        video_path = row["video"]
        clip_json_path = row["clip_json_path"]

        with open(clip_json_path, "r", encoding="utf-8") as f:
            sample = json.load(f)

        fps = float(sample["fps"])
        num_frames_in_clip = int(sample["num_frames_in_clip"])

        if self.random_offset_sec:
            offset_sec = random.random() * self.sample_stride_sec
        else:
            offset_sec = 0.0

        frame_indices = get_target_frame_indices_with_offset(
            num_frames_in_clip=num_frames_in_clip,
            fps=fps,
            stride_sec=self.sample_stride_sec,
            offset_sec=offset_sec,
        )

        if len(frame_indices) == 0:
            raise ValueError(f"No sampled frames for video: {video_path}")

        phase_labels = build_phase_labels_from_sample(sample, frame_indices)

        if self.max_frames_per_video is not None:
            frame_indices = frame_indices[:self.max_frames_per_video]
            phase_labels = phase_labels[:self.max_frames_per_video]

        frames = read_frames_rgb(video_path, frame_indices)

        return {
            "video": video_path,
            "video_path": video_path,
            "frames": frames,
            "frame_indices": frame_indices,
            "phase_labels": phase_labels,
            "clip_json_path": clip_json_path,
            "num_frames_in_clip": num_frames_in_clip,
            "offset_sec": offset_sec,
        }


def collate_fn(batch):
    assert len(batch) == 1
    return batch[0]



# Phase recognition model based on Qwen2.5-VL.
# It supports two backbone modes:
# 1) frozen backbone, where only the temporal module and classification head are trained;
# 2) LoRA adaptation, where parameter-efficient adapters are added to the backbone.
# For each sampled frame, the model extracts a visual-text feature, then applies
# a BiLSTM and a linear head to predict frame-wise phase labels.

class TaskAQwen25VL(nn.Module):
    def __init__(
        self,
        model_name,
        num_phase_classes=7,
        use_lora=False,
        freeze_backbone=True,
        lora_r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        backbone_dtype=None,
    ):
        super().__init__()

        self.processor = AutoProcessor.from_pretrained(model_name)
        self.use_lora = use_lora
        self.freeze_backbone = freeze_backbone

        if backbone_dtype is None:
            backbone_dtype = torch.float32 if freeze_backbone else (
                torch.float16 if torch.cuda.is_available() else torch.float32
            )

        backbone = Qwen2_5_VLModel.from_pretrained(
            model_name,
            torch_dtype=backbone_dtype
        )

        if use_lora:
            lora_config = LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                lora_dropout=lora_dropout,
                bias="none",
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            )
            backbone = get_peft_model(backbone, lora_config)
            backbone.print_trainable_parameters()

        if freeze_backbone:
            for p in backbone.parameters():
                p.requires_grad = False
            backbone.eval()

        self.backbone = backbone

        hidden_size = self.backbone.config.text_config.hidden_size

        self.feat_norm = nn.LayerNorm(hidden_size)
        self.temporal = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.phase_head = nn.Linear(hidden_size, num_phase_classes)

    def encode_one_frame(self, frame_rgb):
        device = next(self.phase_head.parameters()).device

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": frame_rgb},
                    {"type": "text", "text": "phase classification"}
                ]
            }
        ]

        inputs = self.processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
            return_tensors="pt"
        )

        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                inputs[k] = v.to(device)

        if self.freeze_backbone:
            with torch.no_grad():
                outputs = self.backbone(**inputs, return_dict=True)
        else:
            outputs = self.backbone(**inputs, return_dict=True)

        hidden = outputs.last_hidden_state.float()
        feat = hidden.mean(dim=1)[0]

        if torch.isnan(feat).any() or torch.isinf(feat).any():
            raise ValueError("feature has NaN/Inf inside encode_one_frame")

        return feat

    def forward(self, frames):
        feats = []
        for frame_rgb in frames:
            feats.append(self.encode_one_frame(frame_rgb))

        feats = torch.stack(feats, dim=0).unsqueeze(0)   # [1, T, D]
        feats = self.feat_norm(feats)

        if torch.isnan(feats).any() or torch.isinf(feats).any():
            raise ValueError("feats has NaN/Inf after feat_norm")

        temporal_out, _ = self.temporal(feats)
        phase_logits = self.phase_head(temporal_out)

        if torch.isnan(phase_logits).any() or torch.isinf(phase_logits).any():
            raise ValueError("phase_logits has NaN/Inf")

        return phase_logits


# Training and evaluation utilities for phase recognition:
# - compute frame-wise classification loss,
# - evaluate the model on the test split using loss / accuracy / macro F1,
# - train the model epoch by epoch and save training history and the final checkpoint.

def compute_loss(phase_logits, phase_labels, num_phase_classes=7):
    logits = phase_logits[0]

    valid_indices = [
        i for i, x in enumerate(phase_labels)
        if isinstance(x, int) and 0 <= x < num_phase_classes
    ]

    if len(valid_indices) == 0:
        return None

    valid_logits = logits[valid_indices]
    valid_labels = torch.tensor(
        [phase_labels[i] for i in valid_indices],
        dtype=torch.long,
        device=logits.device
    )

    loss_fn = nn.CrossEntropyLoss()
    return loss_fn(valid_logits, valid_labels)


def evaluate_phase_model(model, loader, num_phase_classes=7):
    model.eval()
    if hasattr(model, "backbone"):
        model.backbone.eval()

    total_loss = 0.0
    steps = 0

    all_gt = []
    all_pred = []

    with torch.no_grad():
        for sample in loader:
            phase_logits = model(sample["frames"])

            loss = compute_loss(
                phase_logits,
                sample["phase_labels"],
                num_phase_classes=num_phase_classes
            )

            if loss is not None:
                total_loss += loss.item()
                steps += 1

            logits = phase_logits[0]  # [T, C]
            pred_ids = logits.argmax(dim=-1).detach().cpu().tolist()

            valid_indices = [
                i for i, x in enumerate(sample["phase_labels"])
                if isinstance(x, int) and 0 <= x < num_phase_classes
            ]

            for i in valid_indices:
                all_gt.append(int(sample["phase_labels"][i]))
                all_pred.append(int(pred_ids[i]))

    test_loss = total_loss / max(steps, 1)
    test_accuracy = accuracy_score(all_gt, all_pred) if len(all_gt) > 0 else 0.0
    test_macro_f1 = f1_score(all_gt, all_pred, average="macro", zero_division=0) if len(all_gt) > 0 else 0.0

    return {
        "test_loss": float(test_loss),
        "test_framewise_accuracy": float(test_accuracy),
        "test_framewise_macro_f1": float(test_macro_f1),
    }


def train(model, train_loader, test_loader, optimizer, save_dir, save_name, epochs=8, num_phase_classes=7):
    os.makedirs(save_dir, exist_ok=True)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    print("trainable params:", sum(p.numel() for p in trainable_params))

    global_step = 0
    history = []

    best_metric = -float("inf")
    best_epoch = -1
    best_state_dict = None
    best_eval_metrics = None
    best_train_loss = None

    for epoch in range(epochs):
        model.train()
        if getattr(model, "freeze_backbone", False):
            model.backbone.eval()

        running_loss = 0.0
        steps = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for sample in pbar:
            global_step += 1

            try:
                phase_logits = model(sample["frames"])
            except Exception as e:
                print("\n[FORWARD FAIL]")
                print("video:", sample["video_path"])
                print("frame_indices:", sample["frame_indices"])
                raise e

            if torch.isnan(phase_logits).any() or torch.isinf(phase_logits).any():
                print("\n[BAD LOGITS]")
                print("video:", sample["video_path"])
                print("frame_indices:", sample["frame_indices"])
                print("phase_labels:", sample["phase_labels"])
                raise ValueError("phase_logits invalid")

            loss = compute_loss(
                phase_logits,
                sample["phase_labels"],
                num_phase_classes=num_phase_classes
            )

            if loss is None:
                print("\n[SKIP] no valid labels:", sample["video_path"])
                continue

            if torch.isnan(loss) or torch.isinf(loss):
                print("\n[BAD LOSS]")
                print("video:", sample["video_path"])
                print("phase_labels:", sample["phase_labels"])
                print("logits min/max:", phase_logits.min().item(), phase_logits.max().item())
                raise ValueError("loss invalid")

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=grad_clip_norm)
            optimizer.step()

            running_loss += loss.item()
            steps += 1

            pbar.set_postfix(
                loss=f"{running_loss / max(steps,1):.4f}",
                last=f"{loss.item():.4f}",
                step=global_step
            )

        train_loss = running_loss / max(steps, 1)

        eval_metrics = evaluate_phase_model(
            model=model,
            loader=test_loader,
            num_phase_classes=num_phase_classes
        )

        epoch_record = {
            "epoch": epoch + 1,
            "train_loss": float(train_loss),
            "test_loss": eval_metrics["test_loss"],
            "test_framewise_accuracy": eval_metrics["test_framewise_accuracy"],
            "test_framewise_macro_f1": eval_metrics["test_framewise_macro_f1"],
        }

        history.append(epoch_record)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"test_loss={eval_metrics['test_loss']:.4f} | "
            f"test_acc={eval_metrics['test_framewise_accuracy']:.4f} | "
            f"test_macro_f1={eval_metrics['test_framewise_macro_f1']:.4f}"
        )

        with open(os.path.join(save_dir, "train_history.json"), "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2, ensure_ascii=False)

        current_metric = eval_metrics["test_framewise_macro_f1"]
        if current_metric > best_metric:
            best_metric = current_metric
            best_epoch = epoch + 1
            best_train_loss = float(train_loss)
            best_eval_metrics = copy.deepcopy(eval_metrics)
            best_state_dict = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

    if best_state_dict is None:
        raise RuntimeError("No valid best checkpoint was selected during training.")

    ckpt = {
        "epoch": best_epoch,
        "model_state_dict": best_state_dict,
        "best_metric_name": "test_framewise_macro_f1",
        "best_metric_value": best_metric,
        "train_loss": best_train_loss,
        "test_loss": best_eval_metrics["test_loss"],
        "test_framewise_accuracy": best_eval_metrics["test_framewise_accuracy"],
        "test_framewise_macro_f1": best_eval_metrics["test_framewise_macro_f1"],
        "train_history": history,
    }

    torch.save(ckpt, os.path.join(save_dir, save_name))

    print(f"Training done. Best epoch: {best_epoch}")
    print(f"Best test_framewise_macro_f1: {best_metric:.4f}")
    print("Saved to:", os.path.join(save_dir, save_name))


In [13]:
# Prepare the training/testing datasets and loaders,
# initialize the phase recognition model and optimizer,
# then start frozen-backbone training.

train_dataset = TaskAPhaseDataset(
    train_jsonl,
    max_frames_per_video=max_frames_per_video,
    num_phase_classes=num_phase_classes,
    validate=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

test_dataset = TaskAPhaseDataset(
    test_jsonl,
    max_frames_per_video=max_frames_per_video,
    num_phase_classes=num_phase_classes,
    validate=True,
    random_offset_sec=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

model = TaskAQwen25VL(
    model_name=model_name,
    num_phase_classes=num_phase_classes,
    use_lora=False,
    freeze_backbone=True,
    backbone_dtype=torch.float32,
).to(device)

trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=lr,
    eps=adam_eps
)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    save_dir=save_dir,
    save_name="qwen25vl_frozen.pt",
    epochs=num_epochs,
    num_phase_classes=num_phase_classes
)


num usable rows: 17
num usable rows: 3


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

trainable params: 25200647


Epoch 1/8:   0%|          | 0/17 [00:00<?, ?it/s]

...

Epoch 6/8 | train_loss=1.2324 | test_loss=1.4076 | test_acc=0.1963 | test_macro_f1=0.1035


Epoch 7/8:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 7/8 | train_loss=1.2229 | test_loss=1.9900 | test_acc=0.1519 | test_macro_f1=0.0393


Epoch 8/8:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 8/8 | train_loss=1.0309 | test_loss=1.0541 | test_acc=0.7407 | test_macro_f1=0.2258
Training done. Best epoch: 8
Best test_framewise_macro_f1: 0.2258
Saved to: /content/drive/MyDrive/phase_recognition/outputs/qwen25vl_frozen.pt


In [ ]:
# Optional LoRA training configuration.
# This block reuses the same training pipeline, but switches the backbone mode
# from frozen-backbone training to LoRA-based adaptation.
# It is currently kept disabled because the available compute/memory budget
# is not sufficient for a stable full run under the current setup.


# lora_r = 8
# lora_alpha = 16
# lora_dropout = 0.05


# model = TaskAQwen25VL(
#         model_name=model_name,
#         num_phase_classes=num_phase_classes,
#         use_lora=True,
#         freeze_backbone=False,
#         lora_r=lora_r,
#         lora_alpha=lora_alpha,
#         lora_dropout=lora_dropout,
#     ).to(device)

# train(
#     model=model,
#     train_loader=train_loader,
#     test_loader=test_loader,
#     optimizer=optimizer,
#     save_dir=save_dir,
#     model_name="qwen25vl_lora.pt",
#     epochs=num_epochs,
#     num_phase_classes=num_phase_classes
# )



In [16]:
# Final inference setup:
# load the test JSONL, the trained checkpoint, and define
# where the structured phase predictions will be saved.

test_jsonl = "/content/drive/MyDrive/phase_recognition/testing/gt_10s.jsonl"
ckpt_path = "/content/drive/MyDrive/phase_recognition/outputs/qwen25vl_frozen.pt"
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
pred_out = "/content/drive/MyDrive/phase_recognition/outputs/finetuned_predictions.jsonl"

num_phase_classes = 7
device = "cuda" if torch.cuda.is_available() else "cpu"

max_frames_per_video = 90


# Run final phase inference on the test split using the trained model,
# and save predictions in the unified evaluation format.
@torch.no_grad()
def run_inference():

    dataset = TaskAPhaseDataset(
        test_jsonl,
        max_frames_per_video=max_frames_per_video,
        num_phase_classes=num_phase_classes,
        validate=False,
        random_offset_sec=False
    )
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)

    model = TaskAQwen25VL(
        model_name=model_name,
        num_phase_classes=num_phase_classes,
        use_lora=False,
        freeze_backbone=True,
        backbone_dtype=torch.float32,
    ).to(device)

    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.eval()

    rows = []

    for i, sample in enumerate(loader):
        print(f"running phase inference {i+1}/{len(dataset)}")

        phase_logits = model(sample["frames"])
        pred_phase_ids = phase_logits.argmax(dim=-1)[0].detach().cpu().tolist()

        out_row = {
            "video": sample["video"],
            "clip_json_path": sample.get("clip_json_path", None),
            "ground_truth": {
                "frame_indices": sample["frame_indices"],
                "phase_labels": sample["phase_labels"]
            },
            "prediction": {
                "frame_indices": sample["frame_indices"],
                "phase_labels": pred_phase_ids
            },
            "raw_prediction": None
        }

        rows.append(out_row)

    with open(pred_out, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("saved:", pred_out)


if __name__ == "__main__":
    run_inference()

num usable rows: 3


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

running phase inference 1/3
running phase inference 2/3
running phase inference 3/3
saved: /content/drive/MyDrive/phase_recognition/outputs/finetuned_predictions.jsonl


# predict evaluation

In [17]:
# Final evaluation utilities for phase recognition:
# - validate the unified prediction format,
# - compute JSON parse success rate and schema validity rate,
# - compute frame-wise phase accuracy and macro F1,
# - save the phase confusion matrix and summary metrics.

phase_map = {
    0: "Preparation",
    1: "Calot triangle dissection",
    2: "Clipping and cutting",
    3: "Gallbladder dissection",
    4: "Gallbladder retraction",
    5: "Cleaning and coagulation",
    6: "Gallbladder packaging",
}


# Check whether a prediction follows the unified frame-wise phase schema.
def schema_valid_phase_prediction(obj, num_phase_classes=7):
    if not isinstance(obj, dict):
        return False

    if "frame_indices" not in obj or "phase_labels" not in obj:
        return False

    frame_indices = obj["frame_indices"]
    phase_labels = obj["phase_labels"]

    if not isinstance(frame_indices, list) or not isinstance(phase_labels, list):
        return False

    if len(frame_indices) != len(phase_labels):
        return False

    for x in frame_indices:
        if not isinstance(x, int):
            return False

    for x in phase_labels:
        if not isinstance(x, int):
            return False
        if x != -1 and not (0 <= x < num_phase_classes):
            return False

    return True

# Save a confusion matrix figure for phase classification results.
def save_confusion_matrix(cm, labels, title, out_path):
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()

    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=45, ha="right")
    plt.yticks(ticks, labels)

    thresh = cm.max() / 2.0 if cm.size > 0 else 0.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j, i, str(cm[i, j]),
                ha="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    plt.ylabel("Ground Truth")
    plt.xlabel("Prediction")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()


# Evaluate a saved prediction JSONL file and export summary metrics and confusion matrix.
def evaluate_phase_jsonl(pred_path, output_dir, num_phase_classes=7):
    os.makedirs(output_dir, exist_ok=True)

    rows = load_jsonl(pred_path)

    num_samples = len(rows)
    parse_ok = 0
    schema_ok = 0

    all_gt = []
    all_pred = []

    for row in rows:
        gt = row.get("ground_truth", None)
        pred = row.get("prediction", None)
        raw_pred = row.get("raw_prediction", None)

        if raw_pred is None:
            parse_ok += 1
        else:
            if safe_parse_json(raw_pred) is not None:
                parse_ok += 1

        if schema_valid_phase_prediction(pred, num_phase_classes=num_phase_classes):
            schema_ok += 1

        if not schema_valid_phase_prediction(gt, num_phase_classes=num_phase_classes):
            continue

        if not schema_valid_phase_prediction(pred, num_phase_classes=num_phase_classes):
            continue

        gt_map = {
            int(fid): int(lbl)
            for fid, lbl in zip(gt["frame_indices"], gt["phase_labels"])
        }
        pred_map = {
            int(fid): int(lbl)
            for fid, lbl in zip(pred["frame_indices"], pred["phase_labels"])
        }

        common_frame_indices = sorted(set(gt_map.keys()) & set(pred_map.keys()))

        for fid in common_frame_indices:
            g = gt_map[fid]
            p = pred_map[fid]
            if g != -1 and p != -1:
                all_gt.append(g)
                all_pred.append(p)

    json_parse_success_rate = parse_ok / num_samples if num_samples > 0 else 0.0
    schema_validity_rate = schema_ok / num_samples if num_samples > 0 else 0.0
    framewise_phase_accuracy = accuracy_score(all_gt, all_pred) if len(all_gt) > 0 else 0.0
    framewise_phase_macro_f1 = f1_score(all_gt, all_pred, average="macro", zero_division=0) if len(all_gt) > 0 else 0.0

    summary = {
        "json_parse_success_rate": json_parse_success_rate,
        "schema_validity_rate": schema_validity_rate,
        "framewise_phase_accuracy": framewise_phase_accuracy,
        "framewise_phase_macro_f1": framewise_phase_macro_f1,
    }

    with open(os.path.join(output_dir, "summary_metrics.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print(summary)

    if len(all_gt) > 0:
        labels = sorted(set(all_gt) | set(all_pred))
        cm = confusion_matrix(all_gt, all_pred, labels=labels)
        label_names = [phase_map.get(x, f"phase_{x}") for x in labels]

        save_confusion_matrix(
            cm,
            label_names,
            "Phase Confusion Matrix",
            os.path.join(output_dir, "phase_confusion_matrix.png")
        )

    print("saved to:", output_dir)

In [18]:
evaluate_phase_jsonl(
    pred_path="/content/drive/MyDrive/phase_recognition/outputs/zeroshot_predictions.jsonl",
    output_dir="/content/drive/MyDrive/phase_recognition/outputs/eval_zeroshot_phase",
    num_phase_classes=7
)

{'json_parse_success_rate': 0.3333333333333333, 'schema_validity_rate': 0.3333333333333333, 'framewise_phase_accuracy': 0.9444444444444444, 'framewise_phase_macro_f1': 0.4857142857142857}
saved to: /content/drive/MyDrive/phase_recognition/outputs/eval_zeroshot_phase


In [19]:
evaluate_phase_jsonl(
    pred_path="/content/drive/MyDrive/phase_recognition/outputs/finetuned_predictions.jsonl",
    output_dir="/content/drive/MyDrive/phase_recognition/outputs/eval_finetuned_phase",
    num_phase_classes=7
)

{'json_parse_success_rate': 1.0, 'schema_validity_rate': 1.0, 'framewise_phase_accuracy': 0.7407407407407407, 'framewise_phase_macro_f1': 0.22577408637873755}
saved to: /content/drive/MyDrive/phase_recognition/outputs/eval_finetuned_phase


# plot

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

RESULTS_DIR = Path("/content/drive/MyDrive/phase_recognition/outputs")
VIS_DIR = RESULTS_DIR / "visualizations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

ZEROSHOT_PRED = RESULTS_DIR / "zeroshot_predictions.jsonl"
FINETUNE_PRED = RESULTS_DIR / "finetuned_predictions.jsonl"


phase_map = {
    0: "Preparation",
    1: "Calot triangle dissection",
    2: "Clipping and cutting",
    3: "Gallbladder dissection",
    4: "Gallbladder retraction",
    5: "Cleaning and coagulation",
    6: "Gallbladder packaging",
}

phase_colors = {
    0: "#4E79A7",
    1: "#F28E2B",
    2: "#E15759",
    3: "#76B7B2",
    4: "#59A14F",
    5: "#EDC948",
    6: "#B07AA1",
    -1: "#BBBBBB",
}

MISSING_COLOR = "#D9D9D9"


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def labels_to_segments(frame_indices, phase_labels):
    if len(frame_indices) == 0:
        return []

    segments = []
    start = int(frame_indices[0])
    current_label = int(phase_labels[0])

    for i in range(1, len(frame_indices)):
        if int(phase_labels[i]) != current_label:
            end = int(frame_indices[i - 1])
            width = end - start + 1
            segments.append((start, width, current_label))
            start = int(frame_indices[i])
            current_label = int(phase_labels[i])

    end = int(frame_indices[-1])
    width = end - start + 1
    segments.append((start, width, current_label))
    return segments


def draw_one_bar(ax, y, height, frame_indices, phase_labels, label):
    segments = labels_to_segments(frame_indices, phase_labels)

    for start, width, phase in segments:
        ax.broken_barh(
            [(start, width)],
            (y, height),
            facecolors=phase_colors.get(int(phase), "#999999")
        )

    ax.text(
        x=ax.get_xlim()[0] - 0.02 * (ax.get_xlim()[1] - ax.get_xlim()[0]),
        y=y + height / 2,
        s=label,
        va="center",
        ha="right",
        fontsize=10,
    )


def draw_missing_bar(ax, y, height, x_min, x_max, label, text="Unavailable"):
    ax.broken_barh(
        [(x_min, x_max - x_min)],
        (y, height),
        facecolors=MISSING_COLOR
    )
    ax.text(
        x=ax.get_xlim()[0] - 0.02 * (ax.get_xlim()[1] - ax.get_xlim()[0]),
        y=y + height / 2,
        s=label,
        va="center",
        ha="right",
        fontsize=10,
    )
    ax.text(
        x=(x_min + x_max) / 2,
        y=y + height / 2,
        s=text,
        va="center",
        ha="center",
        fontsize=9,
        color="black"
    )


def safe_name(video_path):
    return Path(video_path).stem.replace(" ", "_")


def plot_phase_timeline_comparison(zeroshot_row, finetune_row, out_path):
    video_path = zeroshot_row["video"]

    gt = zeroshot_row.get("ground_truth", None)
    zs = zeroshot_row.get("prediction", None)
    ft = finetune_row.get("prediction", None)

    if gt is None:
        print(f"[SKIP] missing ground truth: {video_path}")
        return False

    gt_frames = gt["frame_indices"]
    gt_labels = gt["phase_labels"]

    all_frames = list(gt_frames)

    if zs is not None:
        all_frames += zs["frame_indices"]
    if ft is not None:
        all_frames += ft["frame_indices"]

    if len(all_frames) == 0:
        print(f"[SKIP] empty frames: {video_path}")
        return False

    x_min = min(all_frames)
    x_max = max(all_frames) + 1

    fig, ax = plt.subplots(figsize=(12, 3.2))
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(0, 24)
    ax.set_yticks([])
    ax.set_xlabel("Frame Index")
    ax.set_title(Path(video_path).name)

    draw_one_bar(ax, 16, 4, gt_frames, gt_labels, "Ground Truth")

    if zs is not None:
        draw_one_bar(ax, 10, 4, zs["frame_indices"], zs["phase_labels"], "Zero-shot")
    else:
        draw_missing_bar(ax, 10, 4, x_min, x_max, "Zero-shot", text="Invalid / Missing")

    if ft is not None:
        draw_one_bar(ax, 4, 4, ft["frame_indices"], ft["phase_labels"], "Fine-tuned")
    else:
        draw_missing_bar(ax, 4, 4, x_min, x_max, "Fine-tuned", text="Missing")

    legend_handles = [
        Patch(facecolor=phase_colors[k], label=phase_map[k])
        for k in sorted(phase_map.keys())
    ]
    legend_handles.append(Patch(facecolor=MISSING_COLOR, label="Unavailable / Invalid"))

    ax.legend(handles=legend_handles, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    return True


def main():
    zeroshot_rows = load_jsonl(ZEROSHOT_PRED)
    finetune_rows = load_jsonl(FINETUNE_PRED)

    zs_by_video = {row["video"]: row for row in zeroshot_rows}
    ft_by_video = {row["video"]: row for row in finetune_rows}

    common_videos = sorted(set(zs_by_video.keys()) & set(ft_by_video.keys()))
    print("num common videos:", len(common_videos))

    for video in common_videos:
        out_path = VIS_DIR / f"{safe_name(video)}_phase_timeline.png"
        ok = plot_phase_timeline_comparison(
            zeroshot_row=zs_by_video[video],
            finetune_row=ft_by_video[video],
            out_path=out_path
        )
        if ok:
            print("saved:", out_path)


if __name__ == "__main__":
    main()

# plots

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

RESULTS_DIR = Path("/content/drive/MyDrive/phase_recognition/outputs")
VIS_DIR = RESULTS_DIR / "visualizations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

ZEROSHOT_PRED = RESULTS_DIR / "zeroshot_predictions.jsonl"
FINETUNE_PRED = RESULTS_DIR / "finetuned_predictions.jsonl"


phase_map = {
    0: "Preparation",
    1: "Calot triangle dissection",
    2: "Clipping and cutting",
    3: "Gallbladder dissection",
    4: "Gallbladder retraction",
    5: "Cleaning and coagulation",
    6: "Gallbladder packaging",
}

phase_colors = {
    0: "#4E79A7",
    1: "#F28E2B",
    2: "#E15759",
    3: "#76B7B2",
    4: "#59A14F",
    5: "#EDC948",
    6: "#B07AA1",
    -1: "#BBBBBB",
}

MISSING_COLOR = "#D9D9D9"


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def labels_to_segments(frame_indices, phase_labels):
    if len(frame_indices) == 0:
        return []

    segments = []
    start = int(frame_indices[0])
    current_label = int(phase_labels[0])

    for i in range(1, len(frame_indices)):
        if int(phase_labels[i]) != current_label:
            end = int(frame_indices[i - 1])
            width = end - start + 1
            segments.append((start, width, current_label))
            start = int(frame_indices[i])
            current_label = int(phase_labels[i])

    end = int(frame_indices[-1])
    width = end - start + 1
    segments.append((start, width, current_label))
    return segments


def draw_one_bar(ax, y, height, frame_indices, phase_labels, label):
    segments = labels_to_segments(frame_indices, phase_labels)

    for start, width, phase in segments:
        ax.broken_barh(
            [(start, width)],
            (y, height),
            facecolors=phase_colors.get(int(phase), "#999999")
        )

    ax.text(
        x=ax.get_xlim()[0] - 0.02 * (ax.get_xlim()[1] - ax.get_xlim()[0]),
        y=y + height / 2,
        s=label,
        va="center",
        ha="right",
        fontsize=10,
    )


def draw_missing_bar(ax, y, height, x_min, x_max, label, text="Unavailable"):
    ax.broken_barh(
        [(x_min, x_max - x_min)],
        (y, height),
        facecolors=MISSING_COLOR
    )
    ax.text(
        x=ax.get_xlim()[0] - 0.02 * (ax.get_xlim()[1] - ax.get_xlim()[0]),
        y=y + height / 2,
        s=label,
        va="center",
        ha="right",
        fontsize=10,
    )
    ax.text(
        x=(x_min + x_max) / 2,
        y=y + height / 2,
        s=text,
        va="center",
        ha="center",
        fontsize=9,
        color="black"
    )


def safe_name(video_path):
    return Path(video_path).stem.replace(" ", "_")


def plot_phase_timeline_comparison(zeroshot_row, finetune_row, out_path):
    video_path = zeroshot_row["video"]

    gt = zeroshot_row.get("ground_truth", None)
    zs = zeroshot_row.get("prediction", None)
    ft = finetune_row.get("prediction", None)

    if gt is None:
        print(f"[SKIP] missing ground truth: {video_path}")
        return False

    gt_frames = gt["frame_indices"]
    gt_labels = gt["phase_labels"]

    all_frames = list(gt_frames)

    if zs is not None:
        all_frames += zs["frame_indices"]
    if ft is not None:
        all_frames += ft["frame_indices"]

    if len(all_frames) == 0:
        print(f"[SKIP] empty frames: {video_path}")
        return False

    x_min = min(all_frames)
    x_max = max(all_frames) + 1

    fig, ax = plt.subplots(figsize=(12, 3.2))
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(0, 24)
    ax.set_yticks([])
    ax.set_xlabel("Frame Index")
    ax.set_title(Path(video_path).name)

    draw_one_bar(ax, 16, 4, gt_frames, gt_labels, "Ground Truth")

    if zs is not None:
        draw_one_bar(ax, 10, 4, zs["frame_indices"], zs["phase_labels"], "Zero-shot")
    else:
        draw_missing_bar(ax, 10, 4, x_min, x_max, "Zero-shot", text="Invalid / Missing")

    if ft is not None:
        draw_one_bar(ax, 4, 4, ft["frame_indices"], ft["phase_labels"], "Fine-tuned")
    else:
        draw_missing_bar(ax, 4, 4, x_min, x_max, "Fine-tuned", text="Missing")

    legend_handles = [
        Patch(facecolor=phase_colors[k], label=phase_map[k])
        for k in sorted(phase_map.keys())
    ]
    legend_handles.append(Patch(facecolor=MISSING_COLOR, label="Unavailable / Invalid"))

    ax.legend(handles=legend_handles, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    return True


def main():
    zeroshot_rows = load_jsonl(ZEROSHOT_PRED)
    finetune_rows = load_jsonl(FINETUNE_PRED)

    zs_by_video = {row["video"]: row for row in zeroshot_rows}
    ft_by_video = {row["video"]: row for row in finetune_rows}

    common_videos = sorted(set(zs_by_video.keys()) & set(ft_by_video.keys()))
    print("num common videos:", len(common_videos))

    for video in common_videos:
        out_path = VIS_DIR / f"{safe_name(video)}_phase_timeline.png"
        ok = plot_phase_timeline_comparison(
            zeroshot_row=zs_by_video[video],
            finetune_row=ft_by_video[video],
            out_path=out_path
        )
        if ok:
            print("saved:", out_path)


if __name__ == "__main__":
    main()